<a href="https://colab.research.google.com/github/AnwX73/MOHID-Arabic-Dialect-Normalization/blob/main/code/MOHID_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOHID — AraT5v2 Two-Stage Fine-Tuning

This notebook trains **AraT5v2** for Arabic dialect normalization:

**Dialect Arabic → Modern Standard Arabic (MSA)**

## Training Workflow

1. Prepare the team-created **MOHID dataset**.
2. Prepare clean external **Saudi + Egyptian** dialect-to-MSA pairs from Arabic STS.
3. **Stage 1:** Fine-tune AraT5v2 on the external Saudi + Egyptian data.
4. **Stage 2:** Continue fine-tuning the same model on **MOHID Train** (Saudi + Egyptian + Levantine).
5. Evaluate the final model only on the unseen **MOHID Test** split.
6. Save the final model, tokenizer, predictions, and evaluation metrics.

> **Important:** The `dialect` column is used as metadata only.  
> The model input is `dialect_text`, and the target output is `msa_text`.

In [ ]:
# Install compatible libraries for AraT5v2

!pip install -q "transformers==4.46.3" "sentencepiece>=0.2.0" datasets accelerate evaluate sacrebleu openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# Connect Google Drive to access the project files

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Define the MOHID project paths

import os

project_dir = "/content/drive/MyDrive/MOHID_Arabthon"
mohid_path = os.path.join(project_dir, "MOHID_450_Final_Semantic_Review_AraT5v2.xlsx")
arabic_sts_zip = os.path.join(project_dir, "pone.0272991.s001.zip")
arabic_sts_dir = os.path.join(project_dir, "Arabic_STS")
checkpoints_dir = os.path.join(project_dir, "checkpoints")
final_model_dir = os.path.join(project_dir, "MOHID_Final_Model")
results_dir = os.path.join(project_dir, "results")

os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print("Project folder:", project_dir)

Project folder: /content/drive/MyDrive/MOHID_Arabthon


In [ ]:
# Verify the installed Transformers version and GPU

import transformers
import torch

print("Transformers version:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers version: 4.46.3
GPU available: True
GPU: Tesla T4


## 1. Prepare the MOHID dataset

In [ ]:
# Read the MOHID Excel file

import pandas as pd

df = pd.read_excel(
    mohid_path,
    sheet_name="Training Data"
)

print("Number of rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Number of rows: 450
Columns: ['meaning_id', 'domain', 'dialect', 'dialect_text', 'msa_text']


,meaning_id,domain,dialect,dialect_text,msa_text
0,1,Education,Saudi,ممكن أعرف متى يبدأ التسجيل؟,متى يبدأ التسجيل؟
1,1,Education,Egyptian,ممكن أعرف إمتى يبدأ التسجيل؟,متى يبدأ التسجيل؟
2,1,Education,Levantine,فيني أعرف إمتى بيبلش التسجيل؟,متى يبدأ التسجيل؟
3,2,Education,Saudi,ودي أسجل في هالدورة.,أريد التسجيل في هذه الدورة.
4,2,Education,Egyptian,عايز أسجل في الكورس ده.,أريد التسجيل في هذه الدورة.


In [ ]:
# Check the MOHID dataset for missing values and duplicate rows

print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Missing values:
meaning_id      0
domain          0
dialect         0
dialect_text    0
msa_text        0
dtype: int64

Duplicate rows: 0


In [ ]:
# Split MOHID by meaning_id to prevent data leakage

from sklearn.model_selection import train_test_split

meaning_ids = df["meaning_id"].unique()

mohid_train_ids, temp_ids = train_test_split(
    meaning_ids,
    test_size=0.20,
    random_state=42
)

mohid_val_ids, mohid_test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=42
)

mohid_train_df = df[df["meaning_id"].isin(mohid_train_ids)].reset_index(drop=True)
mohid_val_df = df[df["meaning_id"].isin(mohid_val_ids)].reset_index(drop=True)
mohid_test_df = df[df["meaning_id"].isin(mohid_test_ids)].reset_index(drop=True)

print("MOHID Train:", len(mohid_train_df))
print("MOHID Validation:", len(mohid_val_df))
print("MOHID Test:", len(mohid_test_df))

MOHID Train: 360
MOHID Validation: 45
MOHID Test: 45


In [ ]:
# Verify the MOHID split is balanced by dialect

print("Train:")
print(mohid_train_df["dialect"].value_counts())

print("\nValidation:")
print(mohid_val_df["dialect"].value_counts())

print("\nTest:")
print(mohid_test_df["dialect"].value_counts())

Train:
dialect
Saudi        120
Egyptian     120
Levantine    120
Name: count, dtype: int64

Validation:
dialect
Saudi        15
Egyptian     15
Levantine    15
Name: count, dtype: int64

Test:
dialect
Saudi        15
Egyptian     15
Levantine    15
Name: count, dtype: int64


## 2. Prepare Arabic STS External Data

External dialect-to-MSA data used in **Stage 1**:

- **Saudi Arabic → MSA**
- **Egyptian Arabic → MSA**

**Source:** PLOS ONE — Arabic STS Supporting Data  
https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0272991

Only the Saudi and Egyptian dialect pairs required for MOHID were selected and cleaned before training.

In [ ]:
# Extract the Arabic STS dataset if the folder is not already available

import zipfile

if not os.path.exists(arabic_sts_dir):
    with zipfile.ZipFile(arabic_sts_zip, "r") as zip_ref:
        zip_ref.extractall(arabic_sts_dir)

print("Arabic STS folder:", arabic_sts_dir)

Arabic STS folder: /content/drive/MyDrive/MOHID_Arabthon/Arabic_STS


In [ ]:
# Read the Egyptian and MSA Arabic STS file

egypt_path = os.path.join(
    arabic_sts_dir,
    "Data",
    "Translations to MSA and Egyptian Arabic.xlsx"
)

egypt_raw = pd.read_excel(
    egypt_path,
    sheet_name="sts-test"
)

print("Rows:", len(egypt_raw))
print("Columns:", egypt_raw.columns.tolist())

egypt_raw.head()

Rows: 1379
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'ترجمة العمود B إلى الفصحى', 'ترجمة العمود B إلى العامية المصرية', 'ترجمة العمود C إلى الفصحى', 'ترجمة العمود C إلى العامية المصرية']


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,ترجمة العمود B إلى الفصحى,ترجمة العمود B إلى العامية المصرية,ترجمة العمود C إلى الفصحى,ترجمة العمود C إلى العامية المصرية
0,1,2.5,A girl is styling her hair.,A girl is brushing her hair.,فتاة تصفف شعرها,بنت بتعمل شعرها,فتاة تمشط شعرها,بنت بتسرح شعرها
1,2,3.6,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,مجموعة من الرجال يلعبون كرة القدم على الشاطئ,شوية رجالة بيلعبوا كورة عالشط,مجموعة من الأولاد يلعبون كرة القدم على الشاطئ,شوية ولاد بيلعبوا كورة على الشط
2,3,5.0,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,امرأة تقيس كاحل امرأة أخرى,واحدة ست بتقيس رجل واحدة ست تانية,امرأة تقيس كاحل امرأة أخرى,واحدة ست بتقيس رجل واحدة ست تانية
3,4,4.2,A man is cutting up a cucumber.,A man is slicing a cucumber.,رجل يقطع ثمرة خيار,راجل بيقطع خيارة,رجل يقطع ثمرة خيار إلى شرائح,راجل بيقطع خيارة شرايح
4,5,1.5,A man is playing a harp.,A man is playing a keyboard.,رجل يعزف على القيثارة.,راجل بيعزف عالجيتار,رجل يعزف على بيانو إلكتروني.,راجل بيعزف على البيانو.


In [ ]:
# Build clean Egyptian-to-MSA training pairs

egypt_pairs_1 = pd.DataFrame({
    "dialect_text": egypt_raw.iloc[:, 5],
    "msa_text": egypt_raw.iloc[:, 4]
})

egypt_pairs_2 = pd.DataFrame({
    "dialect_text": egypt_raw.iloc[:, 7],
    "msa_text": egypt_raw.iloc[:, 6]
})

egypt_external_df = pd.concat(
    [egypt_pairs_1, egypt_pairs_2],
    ignore_index=True
)

egypt_external_df["dialect"] = "Egyptian"
egypt_external_df["source"] = "Arabic_STS"

egypt_external_df = (
    egypt_external_df
    .dropna()
    .drop_duplicates(subset=["dialect_text", "msa_text"])
    .reset_index(drop=True)
)

print("Egyptian external pairs:", len(egypt_external_df))

egypt_external_df.head()

Egyptian external pairs: 2694


,dialect_text,msa_text,dialect,source
0,بنت بتعمل شعرها,فتاة تصفف شعرها,Egyptian,Arabic_STS
1,شوية رجالة بيلعبوا كورة عالشط,مجموعة من الرجال يلعبون كرة القدم على الشاطئ,Egyptian,Arabic_STS
2,واحدة ست بتقيس رجل واحدة ست تانية,امرأة تقيس كاحل امرأة أخرى,Egyptian,Arabic_STS
3,راجل بيقطع خيارة,رجل يقطع ثمرة خيار,Egyptian,Arabic_STS
4,راجل بيعزف عالجيتار,رجل يعزف على القيثارة.,Egyptian,Arabic_STS


In [ ]:
# Read the Saudi Arabic STS sheet

saudi_path = os.path.join(
    arabic_sts_dir,
    "Data",
    "Translations to Saudi Arabic.xlsx"
)

saudi_raw = pd.read_excel(
    saudi_path,
    sheet_name="الترجمة السعودية"
)

print("Rows:", len(saudi_raw))
print("Columns:", saudi_raw.columns.tolist())

saudi_raw.head()

Rows: 1379
Columns: ['ترجم+D2+A1:A23+A1:A27+D2+A1:A2+A1:D30', 'ترجمة العمود الأول الى العامية السعودية', 'ترجمة العمود D إلى الفصحى', 'ترجمة العمود الثالث الى العامية السعودية ']


,ترجم+D2+A1:A23+A1:A27+D2+A1:A2+A1:D30,ترجمة العمود الأول الى العامية السعودية,ترجمة العمود D إلى الفصحى,ترجمة العمود الثالث الى العامية السعودية
0,فتاة تصفف شعرها,بنت تزين شعرها.,فتاة تمشط شعرها,بنت تكد شعرها.
1,مجموعة من الرجال يلعبون كرة القدم على الشاطئ,رجاجيل يلعبون كورة على البحر.,مجموعة من الأولاد يلعبون كرة القدم على الشاطئ,عيال يلعبون كورة على البحر.
2,امرأة تقوم بقياس كاحل امرأة أخرى,حرمة قاعدة تقوس رجل حرمة ثانية.,امرأة تقيس كاحل امرأة أخرى,حرمة تقوس رجل حرمة ثانية.
3,رجل يقطع ثمرة خيار,رجال يقطف خيارة.,رجل يقطع ثمرة خيار إلى شرائح.,رجال يقطع خيارة حلقات.
4,رجل يعزف على القيثارة.,رجال يدندن على القيثارة .,رجل يلعب على لوحة المفاتيح.,رجال يلعب بكيبورد الكمبيوتر.


In [ ]:
# Build clean Saudi-to-MSA training pairs

saudi_pairs_1 = pd.DataFrame({
    "dialect_text": saudi_raw.iloc[:, 1],
    "msa_text": saudi_raw.iloc[:, 0]
})

saudi_pairs_2 = pd.DataFrame({
    "dialect_text": saudi_raw.iloc[:, 3],
    "msa_text": saudi_raw.iloc[:, 2]
})

saudi_external_df = pd.concat(
    [saudi_pairs_1, saudi_pairs_2],
    ignore_index=True
)

saudi_external_df["dialect"] = "Saudi"
saudi_external_df["source"] = "Arabic_STS"

saudi_external_df = (
    saudi_external_df
    .dropna()
    .drop_duplicates(subset=["dialect_text", "msa_text"])
    .reset_index(drop=True)
)

print("Saudi external pairs:", len(saudi_external_df))

saudi_external_df.head()

Saudi external pairs: 2678


,dialect_text,msa_text,dialect,source
0,بنت تزين شعرها.,فتاة تصفف شعرها,Saudi,Arabic_STS
1,رجاجيل يلعبون كورة على البحر.,مجموعة من الرجال يلعبون كرة القدم على الشاطئ,Saudi,Arabic_STS
2,حرمة قاعدة تقوس رجل حرمة ثانية.,امرأة تقوم بقياس كاحل امرأة أخرى,Saudi,Arabic_STS
3,رجال يقطف خيارة.,رجل يقطع ثمرة خيار,Saudi,Arabic_STS
4,رجال يدندن على القيثارة .,رجل يعزف على القيثارة.,Saudi,Arabic_STS


In [ ]:
# Combine and check the external Saudi and Egyptian data

external_df = pd.concat(
    [saudi_external_df, egypt_external_df],
    ignore_index=True
)

external_df["dialect_text"] = external_df["dialect_text"].astype(str).str.strip()
external_df["msa_text"] = external_df["msa_text"].astype(str).str.strip()

external_df = external_df[
    (external_df["dialect_text"] != "") &
    (external_df["msa_text"] != "")
].drop_duplicates(
    subset=["dialect_text", "msa_text"]
).reset_index(drop=True)

print("Total external pairs:", len(external_df))
print("\nPairs by dialect:")
print(external_df["dialect"].value_counts())

Total external pairs: 5355

Pairs by dialect:
dialect
Egyptian    2678
Saudi       2677
Name: count, dtype: int64


In [ ]:
# Remove external pairs that overlap with MOHID validation or test inputs

protected_inputs = set(
    pd.concat([
        mohid_val_df["dialect_text"],
        mohid_test_df["dialect_text"]
    ]).astype(str).str.strip()
)

external_df = external_df[
    ~external_df["dialect_text"].isin(protected_inputs)
].reset_index(drop=True)

print("External pairs after overlap check:", len(external_df))

External pairs after overlap check: 5355


In [ ]:
# Split the external data into Stage 1 training and validation sets

external_train_df, external_val_df = train_test_split(
    external_df,
    test_size=0.10,
    random_state=42,
    stratify=external_df["dialect"]
)

external_train_df = external_train_df.reset_index(drop=True)
external_val_df = external_val_df.reset_index(drop=True)

print("External Train:", len(external_train_df))
print("External Validation:", len(external_val_df))

print("\nExternal Train by dialect:")
print(external_train_df["dialect"].value_counts())

External Train: 4819
External Validation: 536

External Train by dialect:
dialect
Egyptian    2410
Saudi       2409
Name: count, dtype: int64


## 3. Load AraT5v2

In [ ]:
# Load the AraT5v2 model and tokenizer

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "UBC-NLP/AraT5v2-base-1024"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("AraT5v2 loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

AraT5v2 loaded successfully!


## 4. Convert and tokenize the datasets

In [ ]:
# Convert the pandas DataFrames into Hugging Face datasets

from datasets import Dataset

external_train_dataset = Dataset.from_pandas(
    external_train_df[["dialect_text", "msa_text"]],
    preserve_index=False
)

external_val_dataset = Dataset.from_pandas(
    external_val_df[["dialect_text", "msa_text"]],
    preserve_index=False
)

mohid_train_dataset = Dataset.from_pandas(
    mohid_train_df[["dialect_text", "msa_text"]],
    preserve_index=False
)

mohid_val_dataset = Dataset.from_pandas(
    mohid_val_df[["dialect_text", "msa_text"]],
    preserve_index=False
)

mohid_test_dataset = Dataset.from_pandas(
    mohid_test_df[["dialect_text", "msa_text"]],
    preserve_index=False
)

print("Datasets are ready!")

Datasets are ready!


In [ ]:
# Tokenize dialect inputs and MSA targets with a clear task instruction

task_prefix = "حوّل اللهجة العربية إلى العربية الفصحى مع الحفاظ على المعنى: "

def preprocess_function(examples):
    inputs = [
        task_prefix + text
        for text in examples["dialect_text"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["msa_text"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_external_train = external_train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_external_val = external_val_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_mohid_train = mohid_train_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_mohid_val = mohid_val_dataset.map(
    preprocess_function,
    batched=True
)

tokenized_mohid_test = mohid_test_dataset.map(
    preprocess_function,
    batched=True
)

print("Tokenization completed!")

Map:   0%|          | 0/4819 [00:00<?, ? examples/s]

Map:   0%|          | 0/536 [00:00<?, ? examples/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

Tokenization completed!


## 5. Stage 1 — Fine-tune on external Saudi + Egyptian data

In [ ]:
# Prepare the shared data collator

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Data collator is ready!")

Data collator is ready!


In [ ]:
# Configure Stage 1 training on the external data

from transformers import Seq2SeqTrainingArguments

stage1_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(checkpoints_dir, "stage1_external"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

print("Stage 1 configuration is ready!")

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Stage 1 configuration is ready!


In [ ]:
# Create the Stage 1 trainer

from transformers import Seq2SeqTrainer

stage1_trainer = Seq2SeqTrainer(
    model=model,
    args=stage1_args,
    train_dataset=tokenized_external_train,
    eval_dataset=tokenized_external_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Stage 1 trainer is ready!")

/tmp/ipykernel_1425/1233216998.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  stage1_trainer = Seq2SeqTrainer(


Stage 1 trainer is ready!


In [ ]:
# Fine-tune AraT5v2 on the external Saudi and Egyptian data

stage1_result = stage1_trainer.train()

print("Stage 1 fine-tuning completed!")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
0,2.320700,1.545528
1,1.879700,1.341859


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Stage 1 fine-tuning completed!


## 6. Stage 2 — Continue fine-tuning on MOHID

In [ ]:
# Configure Stage 2 training on the MOHID dataset

stage2_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(checkpoints_dir, "stage2_mohid"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    weight_decay=0.01,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

print("Stage 2 configuration is ready!")

Stage 2 configuration is ready!


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Create the Stage 2 trainer using the Stage 1 model

stage2_trainer = Seq2SeqTrainer(
    model=model,
    args=stage2_args,
    train_dataset=tokenized_mohid_train,
    eval_dataset=tokenized_mohid_val,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print("Stage 2 trainer is ready!")

/tmp/ipykernel_1425/561637887.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  stage2_trainer = Seq2SeqTrainer(


Stage 2 trainer is ready!


In [ ]:
# Continue fine-tuning the model on MOHID Train

stage2_result = stage2_trainer.train()

print("Stage 2 fine-tuning completed!")

Epoch,Training Loss,Validation Loss
1,2.216400,0.849118
2,1.669800,0.762436
3,1.514700,0.740556
4,1.442500,0.723813
5,1.421500,0.718671


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Stage 2 fine-tuning completed!


In [ ]:
# Save the final MOHID model locally before deleting checkpoints

local_final_model_dir = "/content/MOHID_Final_Model"

stage2_trainer.save_model(local_final_model_dir)
tokenizer.save_pretrained(local_final_model_dir)

print("Final model saved locally!")

Final model saved locally!


In [ ]:
# Copy the final MOHID model from Colab to Google Drive

import shutil

drive_final_model_dir = "/content/drive/MyDrive/MOHID_Arabthon/MOHID_Final_Model"

shutil.copytree(
    local_final_model_dir,
    drive_final_model_dir,
    dirs_exist_ok=True
)

print("Final MOHID model saved to Google Drive!")

Final MOHID model saved to Google Drive!


## 7. Evaluate on the unseen MOHID Test set

In [ ]:
# Generate predictions for the unseen MOHID test dataset

import numpy as np

test_results = stage2_trainer.predict(tokenized_mohid_test)

predictions = test_results.predictions
labels = test_results.label_ids

labels = np.where(
    labels != -100,
    labels,
    tokenizer.pad_token_id
)

predicted_texts = tokenizer.batch_decode(
    predictions,
    skip_special_tokens=True
)

reference_texts = tokenizer.batch_decode(
    labels,
    skip_special_tokens=True
)

print("Test predictions generated:", len(predicted_texts))

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

Test predictions generated: 45


In [ ]:
# Show qualitative MOHID test examples

for i in range(min(10, len(mohid_test_df))):
    print("Dialect:", mohid_test_df.iloc[i]["dialect_text"])
    print("Expected MSA:", reference_texts[i])
    print("MOHID:", predicted_texts[i])
    print("-" * 60)

Dialect: أبي مساعدة في حل الواجب.
Expected MSA: أحتاج إلى مساعدة في حل الواجب.
MOHID: أريد مساعدة في حل الواجب.
------------------------------------------------------------
Dialect: محتاج مساعدة في حل الواجب.
Expected MSA: أحتاج إلى مساعدة في حل الواجب.
MOHID: احتاج مساعدة في حل الواجب.
------------------------------------------------------------
Dialect: بدي مساعدة بحل الواجب.
Expected MSA: أحتاج إلى مساعدة في حل الواجب.
MOHID: أريد مساعدة بحل الواجب.
------------------------------------------------------------
Dialect: أبي أعرف نتيجتي.
Expected MSA: أريد معرفة نتيجتي.
MOHID: أريد معرفة نتيجتي.
------------------------------------------------------------
Dialect: عايز أعرف نتيجتي.
Expected MSA: أريد معرفة نتيجتي.
MOHID: أريد معرفة نتيجتي.
------------------------------------------------------------
Dialect: بدي أعرف نتيجتي.
Expected MSA: أريد معرفة نتيجتي.
MOHID: أريد معرفة نتيجتي.
------------------------------------------------------------
Dialect: يمديني أعدل الطلب؟
Expected MSA: ه

In [ ]:
# Calculate BLEU, chrF, and exact-match scores

import sacrebleu

bleu_score = sacrebleu.corpus_bleu(
    predicted_texts,
    [reference_texts]
).score

chrf_score = sacrebleu.corpus_chrf(
    predicted_texts,
    [reference_texts]
).score

exact_match = np.mean([
    prediction.strip() == reference.strip()
    for prediction, reference in zip(predicted_texts, reference_texts)
]) * 100

print(f"BLEU: {bleu_score:.2f}")
print(f"chrF: {chrf_score:.2f}")
print(f"Exact Match: {exact_match:.2f}%")

BLEU: 42.52
chrF: 65.91
Exact Match: 31.11%


In [ ]:
# Save the final test predictions and evaluation metrics

predictions_df = mohid_test_df[
    ["meaning_id", "domain", "dialect", "dialect_text", "msa_text"]
].copy()

predictions_df["mohid_prediction"] = predicted_texts

predictions_path = os.path.join(
    results_dir,
    "MOHID_Test_Predictions.csv"
)

metrics_path = os.path.join(
    results_dir,
    "MOHID_Test_Metrics.txt"
)

predictions_df.to_csv(
    predictions_path,
    index=False,
    encoding="utf-8-sig"
)

with open(metrics_path, "w", encoding="utf-8") as f:
    f.write(f"BLEU: {bleu_score:.2f}\n")
    f.write(f"chrF: {chrf_score:.2f}\n")
    f.write(f"Exact Match: {exact_match:.2f}%\n")

print("Predictions saved to:", predictions_path)
print("Metrics saved to:", metrics_path)

Predictions saved to: /content/drive/MyDrive/MOHID_Arabthon/results/MOHID_Test_Predictions.csv
Metrics saved to: /content/drive/MyDrive/MOHID_Arabthon/results/MOHID_Test_Metrics.txt


## 8. Test MOHID on new dialect sentences

In [ ]:
# Create a simple function to normalize new dialect sentences

def normalize_to_msa(text):
    input_text = task_prefix + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        num_beams=4
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


In [ ]:
# Test MOHID on several new dialect sentences

test_sentences = [
    "أبي أغير موعد الحجز",
    "ما قدرت أفتح التطبيق من الصباح",
    "عايز أعرف الطلب وصل فين",
    "مش عارف أسجل الدخول",
    "بدي أغير موعد الحجز",
    "ما عم يفتح معي التطبيق"
]

for sentence in test_sentences:
    print("Dialect:", sentence)
    print("MOHID:", normalize_to_msa(sentence))
    print("-" * 50)

Dialect: أبي أغير موعد الحجز
MOHID: أريد تغيير موعد الحجز.
--------------------------------------------------
Dialect: ما قدرت أفتح التطبيق من الصباح
MOHID: تأخر فتح التطبيق من الصباح.
--------------------------------------------------
Dialect: عايز أعرف الطلب وصل فين
MOHID: أريد معرفة الطلب وصل أين؟
--------------------------------------------------
Dialect: مش عارف أسجل الدخول
MOHID: هل يمكنني تسجيل الدخول؟
--------------------------------------------------
Dialect: بدي أغير موعد الحجز
MOHID: أريد تغيير موعد الحجز.
--------------------------------------------------
Dialect: ما عم يفتح معي التطبيق
MOHID: لا يمكنني فتح التطبيق.
--------------------------------------------------
